# NDMI monthly composites (2024-11 and 2024-12, 30 m): Nyeri

**Per calendar month**
1. Sentinel-2 L2A (`S2_SR_HARMONIZED`), scenes with **CLOUDY_PIXEL_PERCENTAGE <= 60** only.
2. Pixel-level cloud, shadow, cirrus, snow, saturated and no-data masking (SCL + s2cloudless, dilated).
3. NDMI = (NIR - SWIR1) / (NIR + SWIR1), NIR = B8, SWIR1 = B11, computed per image, averaged to a 30 m grid,
   then the **per-pixel median of all clear observations in the month**.
4. Gap filling: pixels with no clear observation in the month take the median of clear observations
   within +/-30 days of the month.
5. One GeoTIFF per month: `NDMI_Nyeri_2024_11.tif` and `NDMI_Nyeri_2024_12.tif`
   (band 1 = NDMI, band 2 = QA; NoData = -9999). Saved to `E:\ndvi_ndmi_projects\NDMI\Nyeri`.

| QA | Meaning |
|---|---|
| 0 | Median of clear observations inside the month |
| 2 | Filled from clear observations within +/-30 days of the month |
| 255 | No clear observation available (NoData) |

Re-running resumes: finished months are skipped.


In [1]:
# Run once (first time only), then restart the kernel if pip installed anything new
%pip install earthengine-api geopandas rasterio shapely requests ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [1]:
import csv, html, logging, math, os, random, shutil, threading, time, zipfile
from collections import deque
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from datetime import datetime, timedelta, timezone
from pathlib import Path

import ee
import geopandas as gpd
import rasterio
import requests
from rasterio.merge import merge
from shapely import force_2d
from shapely.geometry import box, mapping
from shapely.validation import make_valid

try:
    import ipywidgets as widgets
    from IPython.display import display
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False
    print("ipywidgets not installed -> run: %pip install ipywidgets  (then restart the kernel) for progress bars")

# ============================== CONFIG ==============================
EE_PROJECT = "tonn-512407"
OUT_DIR = Path(r"H:\Vegetation_indices\NDMI\Nyeri")

SHAPEFILE_PATH = r"C:\Users\locateIT1\Downloads\ken_admin1.shp"
NAME_COLUMN = "adm1_name"                              
COUNTIES = ["Nyeri"]
RUN_COUNTIES = ["Nyeri"]

START_MONTH = "2024-11"
END_MONTH = "2024-12"            # last month to download (None = up to the current month)
INCLUDE_CURRENT_MONTH = False    # current month is partial; delete its files and re-run after the month ends

INDEX = "NDMI"
INDICES = [INDEX]
FORMULA = "NDMI = (NIR-SWIR1)/(NIR+SWIR1); NIR=B8, SWIR1=B11"
SCALE = 30
AGGREGATE_TO_SCALE = True
MAX_SCENE_CLOUD = 60
CLOUD_PROB_MAX = 40
CLOUD_BUFFER_M = 40
USE_PRECHECK = False             # True = ask Earth Engine for the clear fraction before downloading (often hits the memory limit); False = measure it from the downloaded file
MIN_VALID_FRACTION = 0.05        # skip a month if < 5 % of the county has a clear observation (0 = keep all)
FILL_WINDOW_DAYS = 30            # gap-fill window either side of the month

TILE_PIXELS = 750                # smaller download tiles (22.5 km) keep each Earth Engine request light; failing tiles still auto-split
MIN_TILE_M = 5000
SIMPLIFY_M = 20

MAX_WORKERS = 6                  # tiles downloaded in parallel (Earth Engine high-volume endpoint). Lower to 3-4 if you see many 429 / 'too many concurrent' warnings
MAX_RETRIES = 8                  # was 20: a stuck tile used to burn ~40 min before the cooldown
BACKOFF_BASE = 5
BACKOFF_MAX = 60
COOLDOWN_S = 120
NODATA = -9999.0
# ====================================================================

N_BANDS = len(INDICES) + 1
OUT_DIR.mkdir(parents=True, exist_ok=True)
logging.basicConfig(level=logging.INFO, force=True,
                    format="%(asctime)s | %(levelname)-7s | %(message)s",
                    handlers=[logging.StreamHandler(), logging.FileHandler(OUT_DIR / f"run_{'_'.join(RUN_COUNTIES)}.log", encoding="utf-8")])
log = logging.getLogger("rdvi")
logging.getLogger("rasterio._env").setLevel(logging.ERROR)


def month_list():
    # list of (label, first day, first day of next month) from START_MONTH to the last wanted month
    cur = datetime.strptime(START_MONTH, "%Y-%m").date().replace(day=1)
    today = datetime.now(timezone.utc).date()
    last = today.replace(day=1) if INCLUDE_CURRENT_MONTH else (today.replace(day=1) - timedelta(days=1)).replace(day=1)
    if END_MONTH:
        last = min(last, datetime.strptime(END_MONTH, "%Y-%m").date().replace(day=1))
    out = []
    while cur <= last:
        nxt = (cur.replace(day=28) + timedelta(days=4)).replace(day=1)
        out.append((cur.strftime("%Y-%m"), cur.isoformat(), nxt.isoformat()))
        cur = nxt
    return out


MONTHS = month_list()

In [2]:
class TooBig(Exception):
    pass


STATS = {"inflight": 0, "retries": 0, "bytes": 0}
_stat_lock = threading.Lock()


def bump(key, n=1):
    with _stat_lock:
        STATS[key] += n


HEAVY = set()   # tile tags that were too big once; later months go straight to quadrants instead of failing first


def needs_split(msg):
    m = str(msg).lower()
    return any(s in m for s in ("must be less than or equal to", "total request size",
                                "computation timed out", "user memory limit exceeded"))


def init_ee():
    url = "https://earthengine-highvolume.googleapis.com"
    try:
        ee.Initialize(project=EE_PROJECT, opt_url=url)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=EE_PROJECT, opt_url=url)


def retry(fn, what, tries=MAX_RETRIES):
    for k in range(1, tries + 1):
        try:
            return fn()
        except TooBig:
            raise
        except Exception as e:
            if k == tries:
                log.error("%s failed after %d attempts: %s", what, tries, e)
                raise
            bump("retries")
            delay = min(BACKOFF_MAX, BACKOFF_BASE * 2 ** (k - 1)) * random.uniform(0.7, 1.3)
            log.warning("%s attempt %d/%d failed (%s). Retrying in %.0fs", what, k, tries, str(e)[:200], delay)
            time.sleep(delay)


def utm_epsg(lon, lat):
    zone = int((lon + 180) // 6) + 1
    return (32600 if lat >= 0 else 32700) + zone


_ADMIN = None


def load_county(name):
    global _ADMIN
    if _ADMIN is None:
        _ADMIN = gpd.read_file(SHAPEFILE_PATH)
    if NAME_COLUMN:
        names = _ADMIN[NAME_COLUMN].astype(str).str.strip().str.lower()
        gdf = _ADMIN[names == name.strip().lower()]
        if gdf.empty:
            raise ValueError(f"'{name}' not found in column '{NAME_COLUMN}'. "
                             f"Available: {sorted(_ADMIN[NAME_COLUMN].astype(str).unique())}")
    else:
        gdf = _ADMIN   # Nyeri-only shapefile: use all features
    if gdf.crs is None:
        log.warning("%s: shapefile has no CRS, assuming EPSG:4326", name)
        gdf = gdf.set_crs(4326)
    gdf = gdf.to_crs(4326)
    geom = gdf.geometry.union_all() if hasattr(gdf.geometry, "union_all") else gdf.geometry.unary_union
    geom = make_valid(force_2d(geom))
    c = geom.centroid
    epsg = utm_epsg(c.x, c.y)
    geom_utm = gpd.GeoSeries([geom], crs=4326).to_crs(epsg).iloc[0].simplify(SIMPLIFY_M, preserve_topology=True)
    geom_ll = gpd.GeoSeries([geom_utm], crs=epsg).to_crs(4326).iloc[0]
    roi = ee.Geometry(mapping(geom_ll), None, False)
    log.info("%s loaded: EPSG:%d, UTM bounds = %s", name, epsg, [round(v) for v in geom_utm.bounds])
    return {"name": name, "roi": roi, "geom_utm": geom_utm, "epsg": epsg}


def tile_grid(geom_utm):
    T = TILE_PIXELS * SCALE
    minx, miny, maxx, maxy = geom_utm.bounds
    x0, y0 = math.floor(minx / T) * T, math.floor(miny / T) * T
    tiles = []
    for r, y in enumerate(range(y0, math.ceil(maxy), T)):
        for c, x in enumerate(range(x0, math.ceil(maxx), T)):
            inter = box(x, y, x + T, y + T).intersection(geom_utm)
            if inter.is_empty:
                continue
            a, b, c2, d = inter.bounds
            bb = (math.floor(a / SCALE) * SCALE, math.floor(b / SCALE) * SCALE,
                  math.ceil(c2 / SCALE) * SCALE, math.ceil(d / SCALE) * SCALE)
            tiles.append((f"r{r}_c{c}", bb))
    return tiles

In [3]:
# ------------------------ Earth Engine: NDMI + cloud mask + monthly composite + gap fill ------------------------
def compute_indices(img, epsg):
    scl = img.select("SCL")
    bad = (scl.eq(0).Or(scl.eq(1)).Or(scl.eq(3)).Or(scl.eq(8))
           .Or(scl.eq(9)).Or(scl.eq(10)).Or(scl.eq(11)))     # nodata, saturated, shadow, cloud, cirrus, snow
    cp = ee.Image(ee.Algorithms.If(img.get("cloud_prob"), img.get("cloud_prob"),
                                   ee.Image.constant(0).rename("probability"))).select("probability")
    bad = bad.Or(cp.gt(CLOUD_PROB_MAX)).focal_max(radius=CLOUD_BUFFER_M, kernelType="circle", units="meters")

    r = img.select(["B8", "B11"]).divide(10000)
    nir, swir = r.select("B8"), r.select("B11")

    # NDMI (Gao 1996): (NIR - SWIR1) / (NIR + SWIR1)
    idx = nir.subtract(swir).divide(nir.add(swir)).rename(INDEX)
    valid = bad.Not().And(nir.gt(0)).And(swir.gt(0))
    out = idx.toFloat().updateMask(valid)

    if AGGREGATE_TO_SCALE:
        out = (out.setDefaultProjection(img.select("B8").projection())
                  .reduceResolution(reducer=ee.Reducer.mean(), bestEffort=False, maxPixels=1024)
                  .reproject(crs=f"EPSG:{epsg}", scale=SCALE))
    return out.set("system:time_start", img.get("system:time_start"))


def prepare_collection(roi, epsg):
    start = ee.Date(MONTHS[0][1]).advance(-FILL_WINDOW_DAYS, "day")
    end = ee.Date(MONTHS[-1][2]).advance(FILL_WINDOW_DAYS, "day")
    s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
          .filterBounds(roi).filterDate(start, end)
          .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", MAX_SCENE_CLOUD)))
    prob = (ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")
            .filterBounds(roi).filterDate(start, end))
    joined = ee.Join.saveFirst("cloud_prob").apply(
        primary=s2, secondary=prob,
        condition=ee.Filter.equals(leftField="system:index", rightField="system:index"))
    return ee.ImageCollection(joined).map(lambda im: compute_indices(ee.Image(im), epsg))


def build_month_image(col, roi, m0, m1):
    d0, d1 = ee.Date(m0), ee.Date(m1)
    blank = ee.Image.constant(0).rename(INDEX).toFloat().updateMask(ee.Image.constant(0))
    safe = lambda c: ee.ImageCollection(c.select(INDEX).merge(ee.ImageCollection([blank])))   # avoids empty-collection errors

    comp = safe(col.filterDate(d0, d1)).median()                                                  # clear observations in the month
    ext = safe(col.filterDate(d0.advance(-FILL_WINDOW_DAYS, "day"), d1.advance(FILL_WINDOW_DAYS, "day"))).median()

    filled = ee.ImageCollection([ext, comp]).mosaic().rename(INDEX)     # month value wins, window value fills gaps
    qa = ee.Image.constant(255).where(ext.mask(), 2).where(comp.mask(), 0).rename("QA")

    out = ee.Image.cat([filled, qa]).toFloat().clip(roi).unmask(NODATA, False)
    return out, comp

In [4]:
# ------------------------------ downloading with verification ------------------------------
def _zip_to_multiband(zpath):
    tmpd = zpath.parent / (zpath.name + "_unz")
    with zipfile.ZipFile(zpath) as z:
        z.extractall(tmpd)
    arrays, prof = [], None
    order = INDICES + ["QA"]
    key = lambda p: order.index(p.stem.split(".")[-1]) if p.stem.split(".")[-1] in order else 99
    for tf in sorted(tmpd.glob("*.tif"), key=key):
        with rasterio.open(tf) as s:
            arrays.append(s.read(1))
            prof = s.profile
    shutil.rmtree(tmpd, ignore_errors=True)
    prof.update(driver="GTiff", count=len(arrays), dtype="float32")
    with rasterio.open(zpath, "w", **prof) as d:
        for i, a in enumerate(arrays, 1):
            d.write(a.astype("float32"), i)


def validate_tif(path, w=None, h=None):
    with rasterio.open(path) as s:
        if s.count != N_BANDS:
            raise IOError(f"expected {N_BANDS} bands, got {s.count}")
        if w is not None and (abs(s.width - w) > 1 or abs(s.height - h) > 1):
            raise IOError(f"unexpected size {s.width}x{s.height}, expected {w}x{h}")
        s.read()


def fetch(url, dest, w, h):
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_name(f"{dest.name}.{os.getpid()}_{random.randint(0, 99999)}.part")
    try:
        with requests.get(url, stream=True, timeout=(30, 300)) as r:
            if r.status_code != 200:
                msg = r.text[:500]
                if needs_split(msg):
                    raise TooBig(msg)
                raise RuntimeError(f"HTTP {r.status_code}: {msg}")
            expected = r.headers.get("Content-Length")
            got = 0
            with open(part, "wb") as f:
                for chunk in r.iter_content(1 << 20):
                    f.write(chunk)
                    got += len(chunk)
                    bump("bytes", len(chunk))
            if expected and not r.headers.get("Content-Encoding") and got != int(expected):
                raise IOError(f"incomplete download {got}/{expected} bytes")
        with open(part, "rb") as f:
            if f.read(4) == b"PK\x03\x04":
                _zip_to_multiband(part)
        validate_tif(part, w, h)
        os.replace(part, dest)
    except BaseException:
        part.unlink(missing_ok=True)
        raise


def download_bounds(image, bounds, epsg, dest):
    xmin, ymin, xmax, ymax = bounds
    w, h = round((xmax - xmin) / SCALE), round((ymax - ymin) / SCALE)
    region = ee.Geometry.Rectangle([xmin + 1, ymin + 1, xmax - 1, ymax - 1], proj=f"EPSG:{epsg}", geodesic=False)

    def attempt():
        try:
            url = image.getDownloadURL({"region": region, "crs": f"EPSG:{epsg}",
                                        "scale": SCALE, "format": "GEO_TIFF"})
        except ee.EEException as e:
            if needs_split(e):
                raise TooBig(str(e))
            raise
        fetch(url, dest, w, h)

    retry(attempt, f"tile {dest.stem} ({w}x{h}px)")


def download_region(image, bounds, epsg, geom_utm, tmpdir, tag):
    dest = tmpdir / f"{tag}.tif"
    if dest.exists():
        return [dest]
    xmin, ymin, xmax, ymax = bounds
    can_split = min(xmax - xmin, ymax - ymin) >= 2 * MIN_TILE_M
    if not (tag in HEAVY and can_split):          # known-heavy tiles skip the doomed full-size request
        try:
            download_bounds(image, bounds, epsg, dest)
            return [dest]
        except TooBig as e:
            if not can_split:
                raise
            HEAVY.add(tag)
            log.warning("Tile %s too large/heavy (%s) -> splitting into quadrants", tag, str(e)[:120])
    xm = round((xmin + xmax) / 2 / SCALE) * SCALE
    ym = round((ymin + ymax) / 2 / SCALE) * SCALE
    quads = [(xmin, ymin, xm, ym), (xm, ymin, xmax, ym), (xmin, ym, xm, ymax), (xm, ym, xmax, ymax)]
    out = []
    for i, q in enumerate(quads):
        if q[2] > q[0] and q[3] > q[1] and box(*q).intersects(geom_utm):
            out += download_region(image, q, epsg, geom_utm, tmpdir, f"{tag}_{i}")
    return out


def merge_tiles(paths, final, label, valid_frac):
    # returns the clear-observation fraction; returns None (nothing written) if the local check finds too little clear data
    srcs = [rasterio.open(p) for p in paths]
    try:
        mosaic, transform = merge(srcs, nodata=NODATA)
        crs = srcs[0].crs
    finally:
        for s in srcs:
            s.close()
    qa = mosaic[1]
    inside = (qa >= 0).sum()
    local_frac = float((qa == 0).sum() / inside) if inside else 0.0
    if valid_frac is None:
        valid_frac = local_frac
        if local_frac < MIN_VALID_FRACTION:
            return None
    profile = dict(driver="GTiff", height=mosaic.shape[1], width=mosaic.shape[2], count=mosaic.shape[0],
                   dtype="float32", crs=crs, transform=transform, nodata=NODATA, compress="LZW",
                   tiled=True, blockxsize=256, blockysize=256, BIGTIFF="IF_SAFER")
    part = final.with_name(final.name + ".part")
    with rasterio.open(part, "w", **profile) as dst:
        dst.write(mosaic.astype("float32"))
        dst.set_band_description(1, INDEX)
        dst.set_band_description(2, "QA")
        dst.update_tags(month=label, observed_valid_fraction=f"{valid_frac:.3f}",
                        formula=FORMULA, composite="monthly median of clear observations",
                        resolution_m=SCALE, aggregated_to_scale=AGGREGATE_TO_SCALE,
                        fill_window_days=FILL_WINDOW_DAYS, max_scene_cloud=MAX_SCENE_CLOUD,
                        qa_legend="0=clear in month;2=filled from +/-window days;255=unfilled")
    validate_tif(part)
    os.replace(part, final)
    return valid_frac

In [5]:
# ------------------------------ progress dashboard (ipywidgets) ------------------------------
def fmt_t(sec):
    if sec is None or sec != sec:
        return "--"
    sec = int(sec)
    h, r = divmod(sec, 3600)
    m, s = divmod(r, 60)
    return f"{h}h {m:02d}m" if h else (f"{m}m {s:02d}s" if m else f"{s}s")


class Dashboard:
    """Two progress bars + live status + log box. Only the main thread touches the widgets."""

    def __init__(self, county, n_months, todo):
        self.county, self.n_months, self.todo = county, n_months, todo
        self.t0 = time.time()
        self.month_t0 = self.t0
        self.real_done, self.real_secs = 0, 0.0
        self.i, self.label, self.phase = 0, "", "starting"
        self.tiles_done, self.tiles_total = 0, 0
        self.lines = deque(maxlen=12)
        self.last = ""
        if HAVE_WIDGETS:
            lay = widgets.Layout(width="95%")
            st = {"description_width": "60px"}
            self.month_bar = widgets.IntProgress(value=0, min=0, max=max(n_months, 1), description="Months",
                                                 bar_style="info", layout=lay, style=st)
            self.tile_bar = widgets.IntProgress(value=0, min=0, max=1, description="Tiles",
                                                bar_style="info", layout=lay, style=st)
            self.status = widgets.HTML()
            self.logbox = widgets.HTML()
            display(widgets.VBox([self.month_bar, self.tile_bar, self.status, self.logbox]))
        self.render()

    def month_start(self, i, label):
        self.i, self.label, self.phase = i, label, "checking"
        self.month_t0 = time.time()
        self.tiles_done, self.tiles_total = 0, 0
        self.render()

    def month_end(self, i, status):
        if status != "exists":
            self.real_done += 1
            self.real_secs += time.time() - self.month_t0
        if HAVE_WIDGETS:
            self.month_bar.value = i
        self.render()

    def set_phase(self, phase, bar_style=None):
        self.phase = phase
        if HAVE_WIDGETS and bar_style:
            self.tile_bar.bar_style = bar_style
        self.render()

    def set_tiles(self, done, total):
        self.tiles_done, self.tiles_total = done, total
        if HAVE_WIDGETS:
            self.tile_bar.max = max(total, 1)
            self.tile_bar.value = done
        self.render()

    def finish(self):
        self.phase = "all done"
        if HAVE_WIDGETS:
            self.month_bar.bar_style = self.tile_bar.bar_style = "success"
        self.render()

    def render(self):
        if not HAVE_WIDGETS:
            return
        now = time.time()
        avg = self.real_secs / self.real_done if self.real_done else None
        left = max(self.todo - self.real_done, 0)
        eta = avg * left if avg is not None else None
        with _stat_lock:
            s = dict(STATS)
        self.status.value = (
            f"<div style='line-height:1.6'><b>{self.county}</b> &middot; month {self.i}/{self.n_months} "
            f"<b>{self.label}</b> &middot; {html.escape(self.phase)}<br>"
            f"tiles {self.tiles_done}/{self.tiles_total} (in flight: {s['inflight']}) &middot; "
            f"downloaded {s['bytes'] / 1e6:,.0f} MB &middot; retries so far: {s['retries']}<br>"
            f"this month {fmt_t(now - self.month_t0)} &middot; avg/month {fmt_t(avg)} &middot; "
            f"months still to do {left} &middot; <b>ETA {fmt_t(eta)}</b> &middot; elapsed {fmt_t(now - self.t0)}</div>")
        self.logbox.value = ("<pre style='font-size:11px;margin:6px 0 0 0;max-height:230px;overflow:auto'>"
                             + html.escape("\n".join(self.lines)) + "</pre>")


class UIHandler(logging.Handler):
    def __init__(self, ui):
        super().__init__()
        self.ui = ui
        self.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))

    def emit(self, record):
        self.ui.lines.append(self.format(record))


UI = None


def cooldown(seconds, why):
    end = time.time() + seconds
    while time.time() < end:
        UI.set_phase(f"{why} - retrying in {int(end - time.time())}s", "danger")
        time.sleep(1)


# ------------------------------ per-month processing & main loop ------------------------------
def _tile_job(image, bounds, county, tmpdir, tag):
    bump("inflight", 1)
    try:
        return download_region(image, bounds, county["epsg"], county["geom_utm"], tmpdir, tag)
    finally:
        bump("inflight", -1)


def process_month(county, col, month, tiles):
    label, m0, m1 = month
    name = county["name"]
    cdir = OUT_DIR / name
    cdir.mkdir(parents=True, exist_ok=True)
    final = cdir / f"{INDEX}_{name}_{label.replace('-', '_')}.tif"
    if final.exists():
        return "exists", None

    image, comp = build_month_image(col, county["roi"], m0, m1)
    frac = None
    try:
        if not USE_PRECHECK:
            raise RuntimeError("pre-check disabled (USE_PRECHECK = False)")
        UI.set_phase("pre-check (clear fraction)")
        v = retry(lambda: comp.mask().reduceRegion(
            reducer=ee.Reducer.mean(), geometry=county["roi"], scale=300,
            maxPixels=1e10, bestEffort=True, tileScale=16).get(INDEX).getInfo(),
            f"{name} {label} valid fraction", tries=2)
        frac = float(v or 0)
    except Exception as e:
        if USE_PRECHECK:
            log.warning("[%s] %s pre-check skipped (%s); will check the clear fraction after download", name, label, str(e)[:120])
    if frac is not None and frac < MIN_VALID_FRACTION:
        return "skipped_cloudy_or_partial", frac

    tmpdir = cdir / "_tmp" / f"{label}_{os.getpid()}"
    tmpdir.mkdir(parents=True, exist_ok=True)

    # ---- tiles downloaded in parallel; the main thread only waits and refreshes the dashboard ----
    UI.set_phase("downloading tiles", "info")
    n = len(tiles)
    UI.set_tiles(0, n)
    paths, first_err, done_n = [], None, 0
    pool = ThreadPoolExecutor(max_workers=MAX_WORKERS)
    try:
        pending = {pool.submit(_tile_job, image, b, county, tmpdir, tag) for tag, b in tiles}
        while pending:
            done, pending = wait(pending, timeout=1.0, return_when=FIRST_COMPLETED)
            for f in done:
                if f.cancelled():
                    continue
                try:
                    paths += f.result()
                    done_n += 1
                except Exception as e:
                    if first_err is None:
                        first_err = e
                        for p in pending:          # stop queued tiles; running ones finish and keep their files
                            p.cancel()
            UI.set_tiles(done_n, n)
            if not HAVE_WIDGETS and done:
                log.info("[%s] %s tiles %d/%d", name, label, done_n, n)
    except KeyboardInterrupt:
        pool.shutdown(wait=False, cancel_futures=True)
        raise
    finally:
        pool.shutdown(wait=True)
    if first_err is not None:
        raise first_err

    UI.set_phase("merging + writing GeoTIFF")
    got = merge_tiles(paths, final, label, frac)
    shutil.rmtree(tmpdir, ignore_errors=True)
    if got is None:
        return "skipped_cloudy_or_partial", 0.0
    return "done", got


def process_month_until_done(county, col, month, tiles):
    rnd = 0
    while True:
        rnd += 1
        try:
            return process_month(county, col, month, tiles)
        except Exception as e:
            log.error("[%s] %s not complete after round %d (%s). Staying on this month; next round in %ds",
                      county["name"], month[0], rnd, str(e)[:200], COOLDOWN_S)
            cooldown(COOLDOWN_S, f"round {rnd} failed")
            try:
                init_ee()
            except Exception as e2:
                log.warning("Re-initialising Earth Engine failed: %s", e2)


def run(counties=None):
    global UI
    init_ee()
    logfile = OUT_DIR / f"download_log_{'_'.join(counties or COUNTIES)}.csv"
    new = not logfile.exists()
    with open(logfile, "a", newline="", encoding="utf-8") as lf:
        w = csv.writer(lf)
        if new:
            w.writerow(["timestamp", "county", "month", "status", "observed_valid_fraction"])
        for name in (counties or COUNTIES):
            county = load_county(name)
            tiles = tile_grid(county["geom_utm"])
            col = prepare_collection(county["roi"], county["epsg"])
            HEAVY.clear()
            todo = sum(not (OUT_DIR / name / f"{INDEX}_{name}_{l.replace('-', '_')}.tif").exists() for l, _, _ in MONTHS)
            UI = Dashboard(name, len(MONTHS), todo)
            if HAVE_WIDGETS:                       # widget log box replaces the plain console handler (file log stays)
                root = logging.getLogger()
                for h in list(root.handlers):
                    if type(h) is logging.StreamHandler:
                        root.removeHandler(h)
                if not any(isinstance(h, UIHandler) for h in root.handlers):
                    root.addHandler(UIHandler(UI))
                else:
                    next(h for h in root.handlers if isinstance(h, UIHandler)).ui = UI
            log.info("%s: %d months (%s to %s), %d download tiles per month, %d workers, %d months still to do",
                     name, len(MONTHS), MONTHS[0][0], MONTHS[-1][0], len(tiles), MAX_WORKERS, todo)
            for i, m in enumerate(MONTHS, 1):
                UI.month_start(i, m[0])
                status, frac = process_month_until_done(county, col, m, tiles)
                log.info("[%s] %d/%d %s -> %s%s", name, i, len(MONTHS), m[0], status,
                         "" if frac is None else f" (clear {frac:.0%})")
                w.writerow([datetime.now().isoformat(timespec="seconds"), name, m[0], status, frac])
                lf.flush()
                UI.month_end(i, status)
            UI.finish()
    log.info("All done.")


In [6]:
run(RUN_COUNTIES)

C:\Users\locateIT1\anaconda3\envs\pest-disease-modelling\Lib\site-packages\pyogrio\core.py:35: RuntimeWarning: Could not detect GDAL data files.  Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()
2026-10-22 10:14:53,442 | INFO    | Nyeri loaded: EPSG:32737, UTM bounds = [233324, 9928704, 311675, 10001707]
